# 02 · Đồ thị giao dịch IBM AML HI-Small

1. Đỉnh dùng **mã ngân hàng dạng số** (`bank_id`), khớp với `HI-Small_accounts.csv`.
2. Cạnh có cả `usd_paid` (phía trả, theo `pay_ccy`) và `usd_recv` (phía nhận, theo `recv_ccy`). `weight` vẫn là `usd_paid`.
3. Ghi chú PS2 sửa theo `Patterns.txt`: chu trình dài 2–12, không bỏ SCC 2 đỉnh.
4. SCC tính cả trên view ACH (view mà PS2 dùng).
5. Kỳ PS1 đọc từ `analysis_period.json` (kỳ chính, không có đuôi), cửa sổ cho phép 24/48/96 giờ.

## 1. Chọn dữ liệu đầu vào

In [1]:
import json
import os
from io import StringIO
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

print("02_build_graph · BẢN 15/11/2026 (bank_id, usd_recv, kỳ chính)", flush=True)
if os.environ.get("HI_SMALL_ROOT"):
    PROJECT_ROOT = Path(os.environ["HI_SMALL_ROOT"])
else:
    PROJECT_ROOT = Path(r"D:\KLTN\IBM_HI_Small_EDA")
EDGE_FILE = PROJECT_ROOT / "data" / "processed" / "hi_small_graph_transactions.parquet"
FX_FILE = PROJECT_ROOT / "reports" / "tables" / "fx_rates.csv"
PERIOD_FILE = PROJECT_ROOT / "reports" / "tables" / "analysis_period.json"
WEIGHT_MODE = "usd"

if WEIGHT_MODE not in {"usd", "original"}:
    raise ValueError('Set WEIGHT_MODE to "usd" or "original".')

edge_path = Path(EDGE_FILE).expanduser().resolve()
if not edge_path.is_file() or not Path(PERIOD_FILE).is_file():
    raise FileNotFoundError(
        f"Thiếu {edge_path.name} hoặc {Path(PERIOD_FILE).name} trong {PROJECT_ROOT}. "
        r"Hãy chạy D:\KLTN\01_eda_hi_small.ipynb (bản 15/11) trước.")

if "db" in globals():
    db.close()
db = duckdb.connect()
db.execute("SET threads = 4")
source = db.read_parquet(str(edge_path))
graph_input_columns = (
    "transaction_id", "timestamp", "src", "dst",
    "from_bank", "from_acct", "to_bank", "to_acct", "from_bank_id", "to_bank_id",
    "amt_paid", "pay_ccy", "amt_received", "recv_ccy", "payment_format",
)
missing = sorted(set(graph_input_columns) - set(source.columns))
if missing:
    raise ValueError(f"Missing columns in the EDA edge file: {missing}. Rerun the updated EDA notebook.")
source_types = dict(zip(source.columns, source.types))
identifier_columns = ("src", "dst", "from_bank", "from_acct", "to_bank", "to_acct")
if any(str(source_types[name]) != "VARCHAR" for name in identifier_columns):
    raise ValueError("Use the EDA Parquet with bank and account identifiers stored as text.")
source.project(", ".join(graph_input_columns)).create_view("graph_transactions")
print(f"Input: {edge_path}")

with open(PERIOD_FILE, encoding="utf-8") as fh:
    period = json.load(fh)
print("Analysis period:", {k: period[k] for k in ("main_start", "dev_end", "main_end", "data_end")})

fx_join = ""
fx_column = ""
if WEIGHT_MODE == "usd":
    fx_path = Path(FX_FILE).expanduser().resolve()
    if not fx_path.is_file():
        raise FileNotFoundError(f"USD rate file not found: {fx_path}")
    fx_rates = pd.read_csv(fx_path, dtype={"currency": str})
    print(f"USD rates: {fx_path}")

    if not {"currency", "usd_per_unit"} <= set(fx_rates.columns):
        raise ValueError("fx_rates.csv must contain currency and usd_per_unit columns.")
    fx_rates["usd_per_unit"] = pd.to_numeric(fx_rates["usd_per_unit"], errors="raise")
    if (fx_rates["currency"].isna().any() or not fx_rates["currency"].is_unique
            or not (np.isfinite(fx_rates["usd_per_unit"]) & fx_rates["usd_per_unit"].gt(0)).all()):
        raise ValueError("Expected one finite, positive USD rate per currency.")
    db.register("graph_fx_rates", fx_rates[["currency", "usd_per_unit"]])
    missing_currencies = db.execute("""
        SELECT DISTINCT c FROM (
            SELECT pay_ccy AS c FROM graph_transactions UNION SELECT recv_ccy FROM graph_transactions
        ) t LEFT JOIN graph_fx_rates fx ON t.c = fx.currency
        WHERE t.c IS NOT NULL AND fx.currency IS NULL ORDER BY c
    """).fetchall()
    if missing_currencies:
        raise ValueError(f"Missing USD rates for: {[currency for currency, in missing_currencies]}")
    fx_join = ("JOIN graph_fx_rates fxp ON t.pay_ccy = fxp.currency "
               "JOIN graph_fx_rates fxr ON t.recv_ccy = fxr.currency")
    fx_column = (", t.amt_paid * fxp.usd_per_unit AS usd_paid"
                 ", t.amt_received * fxr.usd_per_unit AS usd_recv"
                 ", t.amt_paid * fxp.usd_per_unit AS amt_usd")

02_build_graph · BẢN 15/11/2026 (bank_id, usd_recv, kỳ chính)
Input: D:\KLTN\IBM_HI_Small_EDA\data\processed\hi_small_graph_transactions.parquet
Analysis period: {'main_start': '2022-09-01 00:00:00', 'dev_end': '2022-09-06 13:00:00', 'main_end': '2022-09-11 00:00:00', 'data_end': '2022-09-18 16:19:00'}
USD rates: D:\KLTN\IBM_HI_Small_EDA\reports\tables\fx_rates.csv


## 2. Kiểm tra đầu vào và đánh chỉ số đỉnh

Một đỉnh là một tài khoản trong một ngân hàng, `account_key = bank_id + "_" + account_id`, với `bank_id` là mã ngân hàng **dạng số** (bỏ số 0 đầu). Cùng một ngân hàng có thể được viết `010` hoặc `10` trong file gốc; khóa số gộp chúng lại và khớp với cột `Bank ID` của `HI-Small_accounts.csv`. Mã chuỗi gốc vẫn còn trong `from_bank`/`to_bank` của cạnh để đối chiếu.

`node_id` lấy theo thứ tự chữ của `account_key` (không theo thứ tự dòng, vì thứ tự dòng tương quan với nhãn) và dùng chung cho mọi phần đồ thị lọc. `transaction_id` giữ nguyên từ EDA.

In [2]:
edge_count, unique_ids, self_loops, invalid_rows, first_ts, last_ts = db.execute("""
    SELECT count(*), count(DISTINCT transaction_id),
           count(*) FILTER (WHERE src = dst),
           count(*) FILTER (
               WHERE transaction_id IS NULL OR timestamp IS NULL
                  OR src IS NULL OR dst IS NULL
                  OR from_bank_id IS NULL OR from_acct IS NULL
                  OR to_bank_id IS NULL OR to_acct IS NULL
                  OR src <> concat(CAST(from_bank_id AS VARCHAR), '_', from_acct)
                  OR dst <> concat(CAST(to_bank_id AS VARCHAR), '_', to_acct)
                  OR amt_paid IS NULL OR amt_received IS NULL
                  OR NOT isfinite(amt_paid) OR NOT isfinite(amt_received)
                  OR pay_ccy IS NULL OR recv_ccy IS NULL OR payment_format IS NULL
           ),
           min(timestamp), max(timestamp)
    FROM graph_transactions
""").fetchall()[0]

if edge_count == 0 or edge_count != unique_ids or self_loops or invalid_rows:
    raise ValueError(
        f"Check the cleaned EDA edge file: {edge_count:,} rows, {unique_ids:,} unique "
        f"transaction IDs, {self_loops:,} self-loops, {invalid_rows:,} invalid rows."
    )

db.execute("""
    CREATE OR REPLACE TEMP TABLE nodes AS
    SELECT CAST(row_number() OVER (ORDER BY account_key) - 1 AS INTEGER) AS node_id,
           account_key, bank_id, CAST(bank_id AS VARCHAR) AS bank_code, account_id
    FROM (
        SELECT src AS account_key, from_bank_id AS bank_id, from_acct AS account_id
        FROM graph_transactions
        UNION
        SELECT dst AS account_key, to_bank_id AS bank_id, to_acct AS account_id
        FROM graph_transactions
    )
""")
nodes = db.execute("SELECT * FROM nodes ORDER BY node_id").df()
if not nodes["account_key"].is_unique:
    raise ValueError("An account key maps to more than one bank/account pair.")
print(f"Indexed {len(nodes):,} accounts for {edge_count:,} transactions.")

Indexed 422,734 accounts for 4,487,124 transactions.


## 3. Dựng đồ thị đa cạnh có hướng, có thời gian và trọng số

Gọi $G=(V,E,s,d,t,w)$, trong đó $V$ là tập tài khoản và $E$ là tập giao dịch sau tiền xử lý.
Với mỗi cạnh $e$, $s(e)$ là tài khoản gửi, $d(e)$ là tài khoản nhận, $t(e)$ là thời điểm
giao dịch và $w(e)$ là số tiền. Nhiều giao dịch có thể có cùng cặp đầu mút,
kể cả cùng thời điểm. `transaction_id` định danh từng cạnh.

Mỗi cạnh mang hai số tiền quy USD:
- `usd_paid = amt_paid × usd_per_unit(pay_ccy)`: số tiền **rời** tài khoản gửi. Dùng cho đặc trưng phía gửi.
- `usd_recv = amt_received × usd_per_unit(recv_ccy)`: số tiền **đến** tài khoản nhận. Dùng cho đặc trưng phía nhận và cho tỷ lệ giữ lại ở PS3.

`weight = usd_paid` (`amt_usd` giữ lại cho tương thích). Số tiền là **thuộc tính** của cạnh, không phải khoảng cách: không dùng nó cho shortest path. PS1 không dùng trọng số; fan chỉ đếm đối tác phân biệt.

DuckDB biểu diễn $V$ bằng bảng `nodes` và $E$ bằng danh sách cạnh `graph_edges`.
Bảng `directed_pairs` lưu mỗi cặp gửi–nhận một lần để tính degree và SCC.

In [3]:
weight_field = "usd_paid" if WEIGHT_MODE == "usd" else "amt_paid"
weight_unit = "USD (paid side)" if WEIGHT_MODE == "usd" else "Original payment currency (pay_ccy)"

db.execute(f"""
    CREATE OR REPLACE TEMP VIEW graph_edges AS
    WITH indexed_edges AS (
        SELECT s.node_id AS source_id, d.node_id AS target_id,
               t.src, t.dst, t.from_bank, t.to_bank, t.from_bank_id, t.to_bank_id,
               t.transaction_id, t.timestamp,
               CAST(epoch(t.timestamp) AS BIGINT) AS timestamp_s,
               t.amt_paid, t.pay_ccy,
               t.amt_received AS amount_received, t.recv_ccy,
               t.payment_format {fx_column}
        FROM graph_transactions t
        JOIN nodes s ON t.src = s.account_key
        JOIN nodes d ON t.dst = d.account_key
        {fx_join}
    )
    SELECT *, {weight_field} AS weight FROM indexed_edges
""")

built_edges, invalid_weights = db.execute("""
    SELECT count(*), count(*) FILTER (WHERE weight IS NULL OR NOT isfinite(weight))
    FROM graph_edges
""").fetchall()[0]
if built_edges != edge_count or invalid_weights:
    raise ValueError("The edge list must preserve each transaction ID and its finite weight.")

db.execute("""
    CREATE OR REPLACE TEMP TABLE directed_pairs AS
    SELECT DISTINCT source_id, target_id FROM graph_edges
""")
print(f"Built {built_edges:,} weighted temporal edges. Weight unit: {weight_unit}")

Built 4,487,124 weighted temporal edges. Weight unit: USD (paid side)


## 4. Kiểm tra kích thước đồ thị

`transactions` là số cạnh giao dịch. `unique_directed_pairs` là số cặp gửi–nhận phân biệt.
`parallel_edge_count` đếm các cạnh vượt quá cạnh đầu tiên của mỗi cặp.

In [4]:
ordered_pairs = db.execute("SELECT count(*) FROM directed_pairs").fetchall()[0][0]

graph_summary = pd.DataFrame([
    ("directed", True),
    ("weighted", True),
    ("weight_unit", weight_unit),
    ("accounts", len(nodes)),
    ("banks", nodes["bank_id"].nunique()),
    ("transactions", built_edges),
    ("unique_directed_pairs", ordered_pairs),
    ("parallel_edge_count", built_edges - ordered_pairs),
    ("first_transaction", first_ts),
    ("last_transaction", last_ts),
], columns=["metric", "value"])
display(graph_summary)

,metric,value
0,directed,True
1,weighted,True
2,weight_unit,USD (paid side)
3,accounts,422734
4,banks,23378
5,transactions,4487124
6,unique_directed_pairs,647939
7,parallel_edge_count,3839185
8,first_transaction,2022-09-01 00:00:00
9,last_transaction,2022-09-18 16:18:00


## 5. Thống kê degree của toàn mạng

Tính in-degree và out-degree theo số đối tác phân biệt trong `directed_pairs`.
Bậc theo từng hướng có thể bằng 0. Đây là thống kê mô tả từ tất cả phương thức
thanh toán, còn B3 của PS1 lấy từ tập ACH ở bước 7.

In [5]:
pair_indices = db.execute("SELECT source_id, target_id FROM directed_pairs").fetchnumpy()
node_count = len(nodes)

degrees = pd.DataFrame({
    "in_degree": np.bincount(pair_indices["target_id"], minlength=node_count),
    "out_degree": np.bincount(pair_indices["source_id"], minlength=node_count),
})
degree_summary = degrees.describe(percentiles=[0.5, 0.9, 0.99]).round(4)
display(degree_summary)

,in_degree,out_degree
count,422734.0000,422734.0000
mean,1.5327,1.5327
std,2.7180,26.7071
min,0.0000,0.0000
50%,1.0000,1.0000
90%,3.0000,3.0000
99%,7.0000,10.0000
max,545.0000,14230.0000


## 6. Chuẩn bị cấu trúc SCC cho PS2

Mỗi chu trình có hướng nằm gọn trong một thành phần liên thông mạnh (SCC), nên chỉ cần tìm chu trình trong các SCC có ít nhất **2** tài khoản. Không bỏ SCC 2 đỉnh: `HI-Small_Patterns.txt` có chu trình độ dài 2 (A→B→A), và độ dài chu trình trong file trải từ 2 đến 12, không chỉ 3–6.

SCC được tính trên hai cấu trúc:
- **mọi phương thức**: mô tả toàn mạng;
- **ACH**: view mà PS2 dùng. SCC lớn nhất của toàn mạng phần lớn nhờ cạnh không phải ACH, nên lọc theo SCC toàn mạng sẽ giữ lại quá nhiều.

Ở bước áp dụng rule, duyệt ứng viên trên cấu trúc cặp có hướng, rồi dùng các cạnh gốc trong `graph_edges` để kiểm tra từng bước chuyển tiền: timestamp phải tăng nghiêm ngặt theo chiều chu trình. Các giao dịch cùng phút là các cạnh riêng; `transaction_id` chỉ dùng để định danh.

In [6]:
try:
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components
except ImportError:
    print("Component analysis requires SciPy. Run %pip install scipy, then rerun this cell.")
else:
    ach_pair_indices = db.execute("""
        SELECT DISTINCT source_id, target_id FROM graph_edges WHERE payment_format = 'ACH'
    """).fetchnumpy()

    def adjacency(pairs):
        return coo_matrix(
            (np.ones(len(pairs["source_id"]), dtype=np.int8),
             (pairs["source_id"], pairs["target_id"])),
            shape=(node_count, node_count),
        ).tocsr()

    component_rows = []
    for view, pairs in (("all_formats", pair_indices), ("ACH", ach_pair_indices)):
        topology = adjacency(pairs)
        for mode in ("strong", "weak"):
            count, membership = connected_components(topology, directed=True, connection=mode)
            sizes = np.bincount(membership)
            if mode == "strong" and view == "all_formats":
                strong_component_membership = membership
            if mode == "strong" and view == "ACH":
                ach_strong_component_membership = membership
            component_rows.append({
                "view": view,
                "connectivity": mode,
                "components": count,
                "largest_component": int(sizes.max()),
                "nontrivial_components": int(np.count_nonzero(sizes > 1)),
                "accounts_in_nontrivial": int(sizes[sizes > 1].sum()),
            })
    component_summary = pd.DataFrame(component_rows)
    display(component_summary)

,view,connectivity,components,largest_component,nontrivial_components,accounts_in_nontrivial
0,all_formats,strong,402415,17075,2928,23247
1,all_formats,weak,21785,372089,21785,422734
2,ACH,strong,419264,25,3043,6513
3,ACH,weak,234143,23653,50331,238922


## 7. Tạo đồ thị ACH và đối chứng cấu trúc B3 cho PS1

`ps1_ach_edges` giữ từng giao dịch ACH thuộc **kỳ chính** đọc từ `analysis_period.json`, tức $[\mathrm{main\_start}, \mathrm{main\_end})$, không gồm đuôi cuối kỳ. Muốn mô tả toàn kỳ thì đặt `PS1_SPLIT_START = PS1_SPLIT_END = None`. Notebook PS1 tự chia kỳ chính thành dev/test.

Tại thời điểm $t$, fan-in của tài khoản $v$ đếm số tài khoản gửi tới $v$ có ít nhất một giao dịch trong $[t-W,t]$. Fan-out đếm số tài khoản nhận từ $v$ theo cùng điều kiện. $W=24$ và $48$ giờ là cửa sổ chính; $96$ giờ là kiểm tra độ nhạy. Một đối tác chỉ đóng góp một đơn vị, bất kể số giao dịch hay số tiền. Hàm `ps1_window_counts()` minh họa phép đếm cho một tài khoản tại một thời điểm.

`ps1_b3_pairs` gộp các cặp gửi–nhận từ **chính kỳ ACH đó**. B3 là bậc theo số đối tác phân biệt trong cả kỳ, dùng để kiểm tra giá trị của cửa sổ thời gian ở cùng hướng, cùng tập tài khoản và cùng số cảnh báo.

In [7]:
PS1_SPLIT_START = period["main_start"]   # None = from the first transaction
PS1_SPLIT_END = period["main_end"]       # exclusive; None = to the last transaction

split_start = pd.Timestamp(PS1_SPLIT_START) if PS1_SPLIT_START is not None else None
split_end = pd.Timestamp(PS1_SPLIT_END) if PS1_SPLIT_END is not None else None
if any(pd.isna(value) or value.tzinfo is not None
       for value in (split_start, split_end) if value is not None):
    raise ValueError("Use valid, timezone-naive timestamps for the PS1 split.")
if split_start is not None and split_end is not None and split_start >= split_end:
    raise ValueError("PS1_SPLIT_START must precede PS1_SPLIT_END.")

db.sql("""
    SELECT * FROM graph_edges
    WHERE payment_format = 'ACH'
      AND timestamp >= coalesce(?::TIMESTAMP, TIMESTAMP '-infinity')
      AND timestamp < coalesce(?::TIMESTAMP, TIMESTAMP 'infinity')
""", params=[split_start, split_end]).create_view("ps1_ach_edges", replace=True)
db.execute("""
    CREATE OR REPLACE TEMP VIEW ps1_b3_pairs AS
    SELECT DISTINCT source_id, target_id
    FROM ps1_ach_edges
""")

def ps1_window_counts(account_key, window_end, window_hours):
    """Count distinct ACH counterparties for one window within the PS1 split."""
    if window_hours not in (24, 48, 96):
        raise ValueError("window_hours must be 24, 48 or 96.")
    end = pd.Timestamp(window_end)
    if pd.isna(end) or end.tzinfo is not None:
        raise ValueError("Use a valid, timezone-naive window_end.")
    if ((split_start is not None and end < split_start)
            or (split_end is not None and end >= split_end)):
        raise ValueError("window_end must lie in the PS1 split.")
    start = end - pd.Timedelta(hours=window_hours)
    return db.sql("""
        SELECT count(DISTINCT source_id) FILTER (WHERE dst = ?) AS fan_in,
               count(DISTINCT target_id) FILTER (WHERE src = ?) AS fan_out
        FROM ps1_ach_edges
        WHERE timestamp BETWEEN ? AND ? AND (src = ? OR dst = ?)
    """, params=[account_key, account_key, start, end, account_key, account_key]).fetchall()[0]

ach_edges, ach_accounts, b3_pairs = db.execute("""
    WITH pairs AS (SELECT * FROM ps1_b3_pairs)
    SELECT (SELECT count(*) FROM ps1_ach_edges),
           (SELECT count(*) FROM (
               SELECT source_id AS node_id FROM pairs
               UNION
               SELECT target_id AS node_id FROM pairs
           )),
           (SELECT count(*) FROM pairs)
""").fetchall()[0]

print(f"PS1 period: [{split_start}, {split_end})")
print(f"PS1 ACH: {ach_edges:,} transactions; {ach_accounts:,} accounts.")
print(f"PS1 B3: {b3_pairs:,} distinct directed pairs in the same ACH split.")

PS1 period: [2022-09-01 00:00:00, 2022-09-11 00:00:00)
PS1 ACH: 528,731 transactions; 238,704 accounts.
PS1 B3: 207,899 distinct directed pairs in the same ACH split.


## 8. Lấy góc nhìn ngân hàng hoặc consortium cho PS5

Giả định quan sát dùng trong notebook: một ngân hàng thấy mọi giao dịch có tài khoản gửi hoặc nhận thuộc ngân hàng đó. Một consortium gộp phạm vi quan sát của các ngân hàng thành viên. Mỗi giao dịch xuất hiện một lần ngay cả khi hai đầu cùng thuộc nhóm.

Dùng `bank_edge_view([70])` cho một ngân hàng hoặc `bank_edge_view([70, 10404])` cho một nhóm. Mã có thể là số hoặc chuỗi (`"070"` và `70` là một ngân hàng). Hàm trả về một DuckDB relation để lọc hoặc đếm tiếp.

In [ ]:
def bank_edge_view(bank_codes):
    """Return each edge visible to at least one selected bank (codes compared as integers)."""
    codes = [bank_codes] if isinstance(bank_codes, (str, int, np.integer)) else list(bank_codes)
    try:
        ids = [int(str(code).strip()) for code in codes]
    except ValueError as exc:
        raise ValueError("Bank codes must be whole numbers, e.g. 70 or '070'.") from exc
    if not ids:
        raise ValueError("Pass at least one bank code.")
    return db.sql("""
        SELECT * FROM graph_edges
        WHERE list_contains(?::BIGINT[], from_bank_id)
           OR list_contains(?::BIGINT[], to_bank_id)
    """, params=[ids, ids])


## 9. Xuất đồ thị để dùng ở notebook khác

Các bảng DuckDB ở trên có hiệu lực trong kernel hiện tại. Nếu muốn các notebook PS
đọc trực tiếp danh sách đỉnh và cạnh đã dựng, đặt `GRAPH_OUTPUT_DIR` thành một thư mục
trên máy. Giữ `None` khi chỉ cần xem kết quả trong notebook. Hai tệp Parquet xuất ra
chứa cấu trúc đồ thị và thuộc tính của giao dịch.

In [9]:
GRAPH_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "graph"
if GRAPH_OUTPUT_DIR is not None:
    graph_output_dir = Path(GRAPH_OUTPUT_DIR).expanduser().resolve()
    graph_output_dir.mkdir(parents=True, exist_ok=True)
    db.execute("COPY nodes TO ? (FORMAT PARQUET)", [str(graph_output_dir / "graph_nodes.parquet")])
    db.execute("COPY (SELECT * FROM graph_edges ORDER BY transaction_id) TO ? (FORMAT PARQUET)",
               [str(graph_output_dir / "graph_edges.parquet")])
    print(f"Saved graph_nodes.parquet and graph_edges.parquet in {graph_output_dir}")

Saved graph_nodes.parquet and graph_edges.parquet in D:\KLTN\IBM_HI_Small_EDA\data\processed\graph
